In [1]:
from datasets import load_dataset

# Load the train.csv dataset using Hugging Face datasets (no pandas)
dataset = load_dataset('csv', data_files='train.csv')['train']

# Define a function to combine the prompt and column A
def combine_prompt_and_a(example):
    # Concatenate the prompt and A column with a space in between
    example['combined_text'] = f"{example['prompt']} {example['A']}"
    return example

# Use the .map() function to create the new column
dataset = dataset.map(combine_prompt_and_a)

# Get the combined text of the row at index 51
row_51_text = dataset[51]['combined_text']

# Find the exact character length using Python's len() function
char_length = len(row_51_text)
print(f"Character length of combined_text at index 51: {char_length}")

Character length of combined_text at index 51: 614


In [2]:
from transformers import AutoTokenizer

# Initialize the bert-base-uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Look at the tokenizer's configuration properties (specifically vocab_size)
vocab_size = tokenizer.vocab_size
print(f"Vocab size of bert-base-uncased tokenizer: {vocab_size}")

Vocab size of bert-base-uncased tokenizer: 30522


In [3]:
# Extract the exact integer ID assigned to the [SEP] (Separator) token
sep_token_id = tokenizer.sep_token_id
print(f"Integer ID of [SEP] token: {sep_token_id}")

Integer ID of [SEP] token: 102


In [4]:
# Tokenize the entire prompt column simultaneously
# Converting dataset['prompt'] to a list to avoid type checking issues with datasets.arrow_dataset.Column
tokenized_inputs = tokenizer(
    list(dataset['prompt']),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# Print the shape of the resulting input_ids tensor
input_ids_shape = tokenized_inputs['input_ids'].shape
print(f"Shape of resulting input_ids tensor: {input_ids_shape}")

Shape of resulting input_ids tensor: torch.Size([2000, 128])


In [5]:
# A standard bert-base-uncased model has a hidden embedding size of 768 dimensions
# and uses exactly 12 attention heads in each layer.
hidden_size = 768
num_attention_heads = 12

# Calculate the dimensionality (size) of each individual attention head
head_dim = hidden_size // num_attention_heads
print(f"Dimensionality of each individual attention head: {head_dim}")

Dimensionality of each individual attention head: 64


In [6]:
import torch
from transformers import AutoModel

# Load the bert-base-uncased model using AutoModel.from_pretrained()
model = AutoModel.from_pretrained("bert-base-uncased")

# Tokenize prompt from row ID 0 using the tokenizer's default settings
prompt_0 = dataset[0]['prompt']
inputs = tokenizer(prompt_0, return_tensors='pt')

# Pass this tokenized input through the model
with torch.no_grad():
    outputs = model(**inputs)

# Look at the exact shape of the last_hidden_state tensor
last_hidden_state_shape = outputs.last_hidden_state.shape
print(f"Shape of the last_hidden_state tensor: {last_hidden_state_shape}")

Shape of the last_hidden_state tensor: torch.Size([1, 31, 768])


In [7]:
# Extract the embedding vector representing the [CLS] token (token index 0)
cls_vector = outputs.last_hidden_state[0, 0]

# Sum the first 5 float values of this [CLS] vector
first_5_sum = torch.sum(cls_vector[:5]).item()

# Round the sum to 4 decimal places
rounded_sum = round(first_5_sum, 4)
print(f"Sum of the first 5 float values of the CLS vector: {rounded_sum}")

Sum of the first 5 float values of the CLS vector: -1.2001


In [8]:
# Load bert-base-uncased model with parameter output_attentions=True
model_att = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

# Tokenize the exact string and pass it through the model
text = "Light-ion fusion is a technique."
inputs_att = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs_att = model_att(**inputs_att)

# Extract the attention matrix for the last layer (index -1) and first attention head (index 0)
last_layer_att = outputs_att.attentions[-1]
first_head_att = last_layer_att[0, 0] # shape [seq_len, seq_len]

# Find the token index for 'fusion'
tokens = tokenizer.convert_ids_to_tokens(inputs_att['input_ids'][0])
fusion_idx = tokens.index("fusion")

# Extract attention weight [CLS] (index 0) pays to 'fusion' (index fusion_idx)
attention_weight = first_head_att[0, fusion_idx].item()
rounded_weight = round(attention_weight, 4)
print(f"Attention weight that [CLS] pays to 'fusion': {rounded_weight}")

Attention weight that [CLS] pays to 'fusion': 0.1025


In [9]:
from sentence_transformers import SentenceTransformer, util

# Initialize the sentence-transformers/all-MiniLM-L6-v2 model
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings for both the prompt and Option B for row ID 0
prompt_emb = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
option_b_emb = st_model.encode(dataset[0]['B'], convert_to_tensor=True)

# Calculate the cosine similarity between these two vectors
similarity_score = util.cos_sim(prompt_emb, option_b_emb).item()
rounded_similarity = round(similarity_score, 4)
print(f"Cosine Similarity score: {rounded_similarity}")

Cosine Similarity score: 0.7658


In [10]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Pipeline 1: TF-IDF ranking
combined_texts = []
for row in dataset:
    combined = " ".join([
        str(row['prompt']),
        str(row['A']),
        str(row['B']),
        str(row['C']),
        str(row['D']),
        str(row['E'])
    ])
    combined_texts.append(combined)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

options_cols = ['A', 'B', 'C', 'D', 'E']
tfidf_top_3_list = []

for row in dataset:
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    similarities = []
    for opt in options_cols:
        opt_vec = vectorizer.transform([str(row[opt])])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append(sim)
        
    similarities = np.array(similarities)
    sorted_indices = np.argsort(-similarities, kind='stable')
    top_3_preds = [options_cols[i] for i in sorted_indices[:3]]
    tfidf_top_3_list.append(top_3_preds)

# Pipeline 2: sentence-transformers ranking
prompts = dataset['prompt']
opt_a = dataset['A']
opt_b = dataset['B']
opt_c = dataset['C']
opt_d = dataset['D']
opt_e = dataset['E']

prompt_embs = st_model.encode(prompts, convert_to_tensor=True, batch_size=128)
a_embs = st_model.encode(opt_a, convert_to_tensor=True, batch_size=128)
b_embs = st_model.encode(opt_b, convert_to_tensor=True, batch_size=128)
c_embs = st_model.encode(opt_c, convert_to_tensor=True, batch_size=128)
d_embs = st_model.encode(opt_d, convert_to_tensor=True, batch_size=128)
e_embs = st_model.encode(opt_e, convert_to_tensor=True, batch_size=128)

minilm_top_3_list = []
minilm_scores = []

def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(1, k)

for i in range(len(dataset)):
    p_emb = prompt_embs[i]
    opts_embs = torch.stack([
        a_embs[i],
        b_embs[i],
        c_embs[i],
        d_embs[i],
        e_embs[i]
    ])
    
    similarities = util.cos_sim(p_emb, opts_embs)[0]
    similarities = similarities.cpu().numpy()
    
    sorted_indices = np.argsort(-similarities, kind='stable')
    top_3_preds = [options_cols[idx] for idx in sorted_indices[:3]]
    minilm_top_3_list.append(top_3_preds)
    
    actual_ans = dataset[i]['answer']
    minilm_scores.append(apk(actual_ans, top_3_preds, k=3))

mean_map3_minilm = np.mean(minilm_scores)
print(f"Final MAP@3 score of MiniLM pipeline: {mean_map3_minilm:.6f}")

# Count differences
diff_count = 0
for i in range(len(dataset)):
    actual_ans = dataset[i]['answer']
    tfidf_top_3 = tfidf_top_3_list[i]
    minilm_top_3 = minilm_top_3_list[i]
    
    not_in_tfidf = (actual_ans not in tfidf_top_3)
    in_minilm = (actual_ans in minilm_top_3)
    
    if not_in_tfidf and in_minilm:
        diff_count += 1

print("Number of questions where answer is NOT in TF-IDF Top-3 BUT IS in MiniLM Top-3:", diff_count)

Final MAP@3 score of MiniLM pipeline: 0.423083
Number of questions where answer is NOT in TF-IDF Top-3 BUT IS in MiniLM Top-3: 502


In [11]:
from transformers import pipeline

# Initialize the Hugging Face zero-shot-classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Get prompt and candidate options for row index 1
prompt_1 = dataset[1]['prompt']
candidate_options = [dataset[1]['A'], dataset[1]['B'], dataset[1]['C']]

# Run classification
zero_shot_res = classifier(prompt_1, candidate_labels=candidate_options)

# Probability score given to the top-ranked option
top_score = zero_shot_res['scores'][0]
print(f"Probability score of top-ranked option: {top_score:.4f}")

Probability score of top-ranked option: 0.4575


In [12]:
# Run same zero-shot classification with multi_label=True (uses independent sigmoids)
zero_shot_res_multi = classifier(prompt_1, candidate_labels=candidate_options, multi_label=True)

# Calculate sum of probabilities under both settings
sum_softmax = sum(zero_shot_res['scores'])
sum_sigmoid = sum(zero_shot_res_multi['scores'])

# Compute absolute difference
absolute_diff = abs(sum_softmax - sum_sigmoid)
print(f"Absolute difference of probability sums: {absolute_diff:.4f}")

Absolute difference of probability sums: 0.9995


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load flan-t5-small model and tokenizer
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Construct prompt string for row index 0
prompt_t5 = f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} or B: {dataset[0]['B']}? Answer with just the letter A or B."

# Generate response
t5_inputs = t5_tokenizer(prompt_t5, return_tensors="pt")
t5_outputs = t5_model.generate(**t5_inputs, max_new_tokens=5)
t5_generated_text = t5_tokenizer.decode(t5_outputs[0], skip_special_tokens=True)
print(f"Generated text: {t5_generated_text}")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 29535.16it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated text: B
